# Celebal Excellence Internship Data Engineer

# Spark Questions

### **Objective:** Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using DataFrames.

### **Steps:**
1. Understand limitations of MapReduce and advantages of Spark (in-memory processing, speed).
2. Learn Spark DataFrame concepts and immutability.
3. Perform data cleaning operations (remove duplicates, handle null values).
4. Apply filtering conditions on datasets (age range, category, region).
5. Use aggregation functions (count, sum, avg, min, max).
6. Group data using `groupBy` and apply conditions on aggregated results.
7. Understand wide transformations and shuffle operations.
8. Modify schema (casting, renaming columns).
9. Handle inconsistent data (nulls, empty values, schema issues).
10. Build a complete data processing pipeline combining cleaning and aggregation.

### **Output:**
Spark code (PySpark/Scala) + query results + brief insights on data processing and transformations.

### **Resources:**
* [https://celebaltech.sharepoint.com/:w:/s/Celebal-LMS/IQCA80CDhtBJTIeBBsbuwgAGAQr7dZ2qpz4wIYC2cjkFJXM?e=cfZGCB](https://celebaltech.sharepoint.com/:w:/s/Celebal-LMS/IQCA80CDhtBJTIeBBsbuwgAGAQr7dZ2qpz4wIYC2cjkFJXM?e=cfZGCB)

In [ ]:
pip install pyspark

In [ ]:
# Import SparkSession
from pyspark.sql import SparkSession

# Import commonly used Spark SQL functions with alias
import pyspark.sql.functions as F

# Import Spark SQL data types with alias
import pyspark.sql.types as T

In [ ]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("Week5_Spark_Assignment") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [ ]:
# Load the Superstore dataset
df = spark.read.csv(
    "Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

In [ ]:
df.show(10, truncate=False)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+----------------------------------------------------------------+--------+--------+--------+--------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name  |Segment  |Country      |City           |State     |Postal Code|Region|Product ID     |Category       |Sub-Category|Product Name                                                    |Sales   |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+----------------------------------------------------------------+--------+--------+--------+--------+
|1     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class  |CG-12520   |Claire Gute  

In [ ]:
print(f"Total Records: {df.count()}")

Total Records: 9994


In [ ]:
print("Columns in the Dataset:")
print(df.columns)

Columns in the Dataset:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [ ]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



## Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.


In [ ]:
# Total records before removing duplicates
print("Total Records Before:", df.count())

# Count occurrences of each Customer ID and Order Date combination
print("Duplicate combinations before removing duplicates:")
df.groupBy("Customer ID", "Order Date") \
  .count() \
  .filter(col("count") > 1) \
  .orderBy(col("count").desc()) \
  .show(truncate=False)

Total Records Before: 9994
Duplicate combinations before removing duplicates:
+-----------+----------+-----+
|Customer ID|Order Date|count|
+-----------+----------+-----+
|SV-20365   |9/20/2017 |14   |
|AC-10615   |9/2/2017  |12   |
|PP-18955   |2/5/2016  |11   |
|WB-21850   |12/11/2016|11   |
|PF-19120   |9/17/2015 |10   |
|NP-18325   |8/9/2015  |10   |
|AG-10270   |9/13/2016 |10   |
|BT-11680   |7/9/2015  |9    |
|GG-14650   |10/31/2014|9    |
|PP-18955   |11/10/2016|9    |
|MP-17965   |4/18/2015 |9    |
|DB-13405   |3/17/2017 |9    |
|GB-14575   |9/21/2015 |9    |
|IM-15070   |12/11/2015|9    |
|SG-20080   |10/31/2015|9    |
|SC-20770   |3/13/2016 |9    |
|KH-16510   |12/8/2017 |9    |
|RP-19390   |9/13/2014 |8    |
|CD-12280   |11/5/2017 |8    |
|NR-18550   |6/3/2017  |8    |
+-----------+----------+-----+
only showing top 20 rows


In [ ]:
# Remove duplicates
df_no_duplicates = df.dropDuplicates(["Customer ID", "Order Date"])

In [ ]:
# Total records after removing duplicates
print("Total Records After:", df_no_duplicates.count())

# Check for duplicate combinations again
print("Duplicate combinations after removing duplicates:")
df_no_duplicates.groupBy("Customer ID", "Order Date") \
                .count() \
                .filter(col("count") > 1) \
                .show(truncate=False)

Total Records After: 4992
Duplicate combinations after removing duplicates:
+-----------+----------+-----+
|Customer ID|Order Date|count|
+-----------+----------+-----+
+-----------+----------+-----+



## Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [ ]:

# Filter data for the West region
west_sales = df.filter(F.col("Region") == "West")

# Group by Category and calculate average Sales
avg_sales = west_sales.groupBy("Category") \
                      .agg(F.avg(F.expr("try_cast(trim(Sales) AS DOUBLE)"))
                      .alias("Average_Sales"))

# Display the result
avg_sales.show(truncate=False)

+---------------+------------------+
|Category       |Average_Sales     |
+---------------+------------------+
|Office Supplies|117.48907552370453|
|Furniture      |360.59540420899896|
|Technology     |422.64417449664415|
+---------------+------------------+



## Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.


In [ ]:
# Count null values in the Category column before filling
print("Null values before filling:")
df.filter(col("Category").isNull()).count()

Null values before filling:


0

In [ ]:
# Fill null values in the Category column with "Unknown"
df_filled = df.na.fill({"Category": "Unknown"})

In [ ]:
# Count null values after filling
print("Null values after filling:")
df_filled.filter(col("Category").isNull()).count()

Null values after filling:


0

In [ ]:
# Display a few records
df_filled.select("Category").show(10, truncate=False)

+---------------+
|Category       |
+---------------+
|Furniture      |
|Furniture      |
|Office Supplies|
|Furniture      |
|Office Supplies|
|Furniture      |
|Office Supplies|
|Technology     |
|Office Supplies|
|Office Supplies|
+---------------+
only showing top 10 rows


## Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [ ]:
#Q6
from pyspark.sql.functions import count

# Count records for each city
city_count = df.groupBy("City") \
               .agg(count("*").alias("Total_Records"))

# Filter cities having more than 100 records
result = city_count.filter(col("Total_Records") > 100)

# Display the resulty
result.show(truncate=False)

+-------------+-------------+
|City         |Total_Records|
+-------------+-------------+
|Springfield  |163          |
|Dallas       |157          |
|Philadelphia |537          |
|Los Angeles  |747          |
|San Francisco|510          |
|San Diego    |170          |
|Detroit      |115          |
|Columbus     |222          |
|Chicago      |314          |
|Seattle      |428          |
|New York City|915          |
|Houston      |377          |
|Jacksonville |125          |
+-------------+-------------+



## Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.


In [ ]:
#Q8
# Filter records where Region is 'West' and Category is 'Technology'
filtered_df = df.filter(
    (col("Region") == "West") &
    (col("Category") == "Technology")
)

# Display the filtered records
filtered_df.show(10, truncate=False)

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+-------------+----------+-----------+------+---------------+----------+------------+--------------------------------------------------------------+-------+--------+--------+--------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name     |Segment    |Country      |City         |State     |Postal Code|Region|Product ID     |Category  |Sub-Category|Product Name                                                  |Sales  |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+-------------+----------+-----------+------+---------------+----------+------------+--------------------------------------------------------------+-------+--------+--------+--------+
|8     |CA-2014-115812|6/9/2014  |6/14/2014 |Standard Class|BH-11710   |Brosina Hoffman   |Consumer 

## Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [ ]:
# Q10
# Convert Order Date to TimestampType and rename it to event_time
df_timestamp = df.withColumn(
    "event_time",
    F.expr("try_to_timestamp(`Order Date`, 'M/d/yyyy')")
).drop("Order Date")

# Display the updated schema
df_timestamp.printSchema()

# Display the first 10 rows
df_timestamp.select("event_time").show(10, truncate=False)

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)
 |-- event_time: timestamp (nullable = true)

+-------------------+
|event_time         |
+-------------------+
|2016-11-08 00:00:00|
|2016-11-08 00:00:00|
|2016-06-12 00:00:00|
|2015-10-1

## Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?


In [ ]:
# Q11
category_sales = df.groupBy("Category") \
                   .agg(F.round(F.sum(F.expr("try_cast(trim(Sales) AS DOUBLE)")), 2)
                   .alias("Total_Sales"))

category_sales.show()

+---------------+-----------+
|       Category|Total_Sales|
+---------------+-----------+
|Office Supplies|  703502.93|
|      Furniture|  733046.86|
|     Technology|  835900.07|
+---------------+-----------+



## Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.


In [ ]:
# Count records before cleaning
print("Total Records Before Cleaning:", df.count())

# Remove rows where Customer ID is null or Customer Name is empty
df_clean = df.filter(
    col("Customer ID").isNotNull() &
    (trim(col("Customer Name")) != "")
)

# Count records after cleaning
print("Total Records After Cleaning:", df_clean.count())


Total Records Before Cleaning: 9994
Total Records After Cleaning: 9970


## Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [ ]:
# Q13
# Calculate minimum, maximum, and average sales
sales_statistics = df.agg(
    F.min(F.expr("try_cast(trim(Sales) AS DOUBLE)")).alias("Minimum_Sales"),
    F.max(F.expr("try_cast(trim(Sales) AS DOUBLE)")).alias("Maximum_Sales"),
    F.round(F.avg(F.expr("try_cast(trim(Sales) AS DOUBLE)")), 2).alias("Average_Sales")
)

# Display the result
sales_statistics.show(truncate=False)

+-------------+-------------+-------------+
|Minimum_Sales|Maximum_Sales|Average_Sales|
+-------------+-------------+-------------+
|0.444        |22638.48     |234.42       |
+-------------+-------------+-------------+



## Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?


In [ ]:
# Q14
from pyspark.sql.functions import to_timestamp, col

# Convert Order Date to TimestampType
df_clean = df.withColumn(
    "Order Date",
    to_timestamp(col("Order Date"), "MM/dd/yyyy")
)

# Display the updated schema
df_clean.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: timestamp (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



## Q15: Write a final processing pipeline that:
1. Filters out duplicates.
2. Fills null prices with 0.
3. Groups by store_id to calculate total revenue.

In [ ]:
# Load the Superstore dataset
df = spark.read.csv(
    "Sample - Superstore.csv",
    header=True,
    inferSchema=True
)


# Step 1: Check total records before removing duplicates
print("Total Records Before Removing Duplicates:", df.count())

# Check how many duplicate records are present
duplicate_count = df.count() - df.dropDuplicates().count()
print("Duplicate Records Found:", duplicate_count)

Total Records Before Removing Duplicates: 9998
Duplicate Records Found: 8


In [ ]:
# Step 2: Remove duplicate records
df_clean = df.dropDuplicates()

# Check total records after removing duplicates
print("Total Records After Removing Duplicates:", df_clean.count())

Total Records After Removing Duplicates: 9990


In [ ]:
# Step 3: Check null values in Sales before filling
null_count_before = df_clean.filter(col("Sales").isNull()).count()
print("Null Values in Sales Before Filling:", null_count_before)

Null Values in Sales Before Filling: 0


In [ ]:
# Step 4: Fill null values in Sales with 0
df_clean = df_clean.na.fill({"Sales": 0})

In [ ]:
# Step 5: Verify null values after filling
null_count_after = df_clean.filter(col("Sales").isNull()).count()
print("Null Values in Sales After Filling:", null_count_after)

Null Values in Sales After Filling: 0


In [ ]:
    # Step 6: Group by Region and calculate total revenue
total_revenue = (
    df_clean
    .groupBy("Region")
    .agg(F.round(F.sum(F.expr("try_cast(trim(Sales) AS DOUBLE)")), 2).alias("Total_Revenue"))
    .orderBy(F.col("Total_Revenue").desc())
)

print("\nTotal Revenue by Region:")
total_revenue.show(truncate=False)


Total Revenue by Region:
+-------+-------------+
|Region |Total_Revenue|
+-------+-------------+
|West   |710071.19    |
|East   |668564.95    |
|Central|497740.53    |
|South  |388968.03    |
+-------+-------------+

